# Parse NCERT PDFs with two Kaggle GPUs
Attach the private inventory notebook output. This notebook validates every source hash, uses MinerU Basic Torch/CUDA on two independent T4 workers, and saves page Markdown and native assets. Set `LIMIT_CHAPTERS=2` for a first check, then 0 for the full run.


In [1]:
%pip -q install 'mineru[torch]==4.1.7' 'docvortex==0.6.13' 'onnxruntime==1.31.1' 'transformers==5.22.2' pymupdf huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib, json, os, shutil, zipfile
from pathlib import Path
import torch
from huggingface_hub import snapshot_download

LIMIT_CHAPTERS = 0
assert torch.cuda.device_count() == 2, 'Enable Kaggle T4 x2'
manifest_paths = list(Path('/kaggle/input').rglob('ncert_inventory/manifest.json'))
archives = list(Path('/kaggle/input').rglob('ncert_inventory/ncert_source_pdfs.zip'))
gap_paths = list(Path('/kaggle/input').rglob('ncert_inventory/gaps.json'))
assert len(manifest_paths) == len(archives) == len(gap_paths) == 1, 'Attach exactly one inventory output'
assert not json.loads(gap_paths[0].read_text()), 'Resolve inventory gaps before the full parse'
manifest = json.loads(manifest_paths[0].read_text())
assert len({r['file'] for r in manifest}) == len(manifest)
assert {(r['grade'], r['subject']) for r in manifest} == {(g, s) for g in range(6, 11) for s in ('maths', 'science')}
assert len({(r['grade'], r['subject'], r['chapter']) for r in manifest}) == len(manifest)
source_root = Path('/kaggle/working/ncert_source')
source_root.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as bundle:
    for row in manifest:
        assert Path(row['file']).name == row['file']
        data = bundle.read(row['file'])
        assert hashlib.sha256(data).hexdigest() == row['sha256']
        (source_root / row['file']).write_bytes(data)
jobs = sorted(manifest, key=lambda r: r['pages'], reverse=True)
if LIMIT_CHAPTERS: jobs = jobs[:LIMIT_CHAPTERS]
all_jobs = jobs
output_root = Path('/kaggle/working/ncert_parsed')
output_root.mkdir(exist_ok=True)
prior_summaries = list(Path('/kaggle/input').rglob('ncert_parsed/summary.json'))
assert len(prior_summaries) <= 1, 'Attach at most one prior parse'
reused = []
if prior_summaries:
    previous = json.loads(prior_summaries[0].read_text())
    assert not previous['errors'], 'Prior parse has errors'
    old_root = prior_summaries[0].parent
    for job in jobs:
        if previous['source_hashes'].get(job['file']) != job['sha256']: continue
        old_chapter = old_root / Path(job['file']).stem
        if not all((old_chapter / f'{page:04d}.done').exists() and
                   (old_chapter / f'{page:04d}.done').read_text().strip() in {'parsed', 'empty'} and
                   (old_chapter / f'{page:04d}' / 'markdown.md').exists()
                   for page in range(1, job['pages'] + 1)): continue
        shutil.copytree(old_chapter, output_root / old_chapter.name)
        reused.append(job)
jobs = [job for job in jobs if job not in reused]
print('Verified', len(manifest), 'source PDFs; reused', len(reused), 'chapters; parsing', len(jobs))

home = Path('/kaggle/working/mineru_home')
model_dir = home / 'models/MinerU-4_models_torch'
snapshot_download('opendatalab/MinerU-4_models_torch', revision='2b3afb86f4d23fa623f6b2f4b2279ed4ef93a89d', local_dir=str(model_dir))
(model_dir / '.mineru_complete').touch()


Verified 130 source PDFs; reused 103 chapters; parsing 27


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

In [3]:
worker = "if __name__ == '__main__':\n    import json, os, sys, traceback\n    from pathlib import Path\n\n    gpu, jobs_file, source_root, output_root, home = sys.argv[1:]\n    os.environ['CUDA_VISIBLE_DEVICES'] = gpu\n    os.environ['MINERU_HOME'] = home\n    os.environ['MINERU_MODEL_SMALL_BACKEND'] = 'torch'\n    os.environ['MINERU_MODEL_SOURCE'] = 'local'\n    os.environ['MINERU_DEVICE_MODE'] = 'cuda'\n    os.environ['MINERU_MODEL_ONNX_DEVICE'] = 'cpu'\n    import torch\n    assert torch.cuda.is_available(), 'CUDA required'\n    from mineru.model.runtime.device import resolve_torch_device\n    assert resolve_torch_device().device.startswith('cuda'), 'MinerU CUDA required'\n    from mineru import MinerUParser\n    from mineru.parser.writer import FileBasedDataWriter\n\n    parser = MinerUParser(tier='basic', parse_mode='auto')\n    jobs = json.loads(Path(jobs_file).read_text())\n    with (Path(output_root) / ('worker-' + gpu + '.jsonl')).open('a') as log:\n        for job in jobs:\n            pdf = Path(source_root) / job['file']\n            chapter = Path(output_root) / Path(job['file']).stem\n            chapter.mkdir(exist_ok=True)\n            for page in range(1, job['pages'] + 1):\n                done = chapter / f'{page:04d}.done'\n                if done.exists():\n                    continue\n                try:\n                    result = parser.parse(str(pdf), page_range=str(page))\n                    assert [p.page_idx + 1 for p in result.pages] == [page]\n                    page_dir = chapter / f'{page:04d}'\n                    page_dir.mkdir(exist_ok=True)\n                    result.save(FileBasedDataWriter(str(page_dir)))\n                    markdown = page_dir / 'markdown.md'\n                    assert markdown.exists()\n                    status = 'empty' if not markdown.read_text(encoding='utf-8').strip() else 'parsed'\n                    done.write_text(status)\n                    log.write(json.dumps({'file': job['file'], 'page': page, 'status': status}) + '\\n')\n                except Exception as exc:\n                    log.write(json.dumps({'file': job['file'], 'page': page, 'status': 'error', 'error': str(exc), 'trace': traceback.format_exc()[-1200:]}) + '\\n')\n                log.flush()\n"
Path('/kaggle/working/parse_worker.py').write_text(worker)


2221

In [4]:
import subprocess, sys
assignments = [[], []]
loads = [0, 0]
for job in jobs:
    gpu = loads.index(min(loads))
    assignments[gpu].append(job)
    loads[gpu] += job['pages']
processes = []
for gpu in range(2):
    job_file = output_root / f'jobs-{gpu}.json'
    job_file.write_text(json.dumps(assignments[gpu]))
    log = (output_root / f'gpu-{gpu}.log').open('w')
    command = [sys.executable, '/kaggle/working/parse_worker.py', str(gpu), str(job_file), str(source_root), str(output_root), str(home)]
    processes.append((subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT), log))
for process, log in processes:
    process.wait()
    log.close()
shutil.rmtree(home)
shutil.rmtree(source_root)
assert all(process.returncode == 0 for process, _ in processes), 'Inspect gpu logs'
rows = [json.loads(line) for gpu in range(2) for line in (output_root / f'worker-{gpu}.jsonl').read_text().splitlines()]
rows.extend({'file': job['file'], 'page': page, 'status': (output_root / Path(job['file']).stem / f'{page:04d}.done').read_text().strip()}
            for job in reused for page in range(1, job['pages'] + 1))
expected = sum(job['pages'] for job in all_jobs)
assert len(rows) == expected and len({(r['file'], r['page']) for r in rows}) == expected
assert all(r['status'] in {'parsed', 'empty', 'error'} for r in rows)
summary = {'chapters': len(all_jobs), 'pages': expected, 'parsed': sum(r['status'] == 'parsed' for r in rows), 'empty': sum(r['status'] == 'empty' for r in rows), 'errors': [r for r in rows if r['status'] == 'error'], 'source_hashes': {r['file']: r['sha256'] for r in all_jobs}}
(output_root / 'summary.json').write_text(json.dumps(summary, indent=2))
print({k: v for k, v in summary.items() if k not in ('errors', 'source_hashes')}, 'errors', len(summary['errors']))
assert not summary['errors'], 'Inspect errors before downstream use'


{'chapters': 130, 'pages': 3066, 'parsed': 3047, 'empty': 19} errors 0
